# Skynode v4: export v3 for the brain, then fine-tune on your phone footage

This notebook sits next to **Skynode v3 Training** and never touches it or its results.
It reads `drone_v3/weights/best.pt` and writes only new things: `skynode/models/*.onnx` and a new run folder `drone_v4/`.

**Part 1 (cells 1-3, about 2 minutes)** exports v3 to the `.onnx` file `brain/detector.py` reads. You can stop there.

**Part 2 (rest)** fine-tunes on your own phone videos, because v3 has only seen internet footage and your sky looks different:
1. Film planes and drones. Upload to **Drive ▸ skynode ▸ phone_videos**. Name each clip after what is in it: `drone_garden.mp4`, `aircraft_morning.mp4`. Mixed or empty-sky clips: `sky_clear.mp4`.
2. v3 itself draws first-guess boxes (much better than a generic model). You check them on contact sheets.
3. It trains v4 from v3's weights (backbone frozen so it doesn't forget what v3 learned), mixing in a sample of your Roboflow dataset as a reminder.
4. It compares v3 and v4 on phone videos the model never trained on.

GPU: Runtime ▸ Change runtime type ▸ T4 GPU. If Colab disconnects, run all cells again: training resumes from `drone_v4/weights/last.pt`.

## 1 · Settings

In [ ]:
PROJECT = "/content/drive/MyDrive/skynode"
V3_WEIGHTS = f"{PROJECT}/drone_v3/weights/best.pt"
CLASSES = ["drone", "aircraft"]              # same order as v3
IMGSZ = 960                                  # v3's training size; keep it
REPO_URL = "https://github.com/bsalsa2/Skynode"
REPO_BRANCH = "claude/colab-training"        # change to "main" once that branch is merged

# ---- Part 2 only ----
PHONE_VIDEOS = f"{PROJECT}/phone_videos"
FRAME_EVERY_S = 1.0
MAX_FRAMES_PER_VIDEO = 120
VAL_FRACTION = 0.2
PRELABEL_CONF = 0.25
NEW_RUN = "drone_v4"
EPOCHS = 25
FREEZE_LAYERS = 10                           # freeze the backbone; small data should nudge, not rewrite
LR0 = 0.002

# Replay: a sample of your Roboflow dataset, so v4 doesn't forget what v3 knew.
# Needs your key as a Colab secret named ROBOFLOW_API_KEY (the key icon in the left sidebar).
# If the download errors, copy the exact rf.workspace(...).project(...).version(...) line from Cell 2 of v3.
USE_REPLAY = True
RF_WORKSPACE = "3-class-airplane-drone-helicopter"
RF_PROJECT = "airplane-drone-helicopter"
RF_VERSION = 2
REPLAY_LIMITS = {"train": 4000, "val": 500}
ALIASES = {                                   # their class name -> ours (None = ignore)
    "multi-rotor": "drone", "fixed wing uav": "drone",
    "civilian aircraft": "aircraft", "military aircraft": "aircraft", "military helicopter": "aircraft",
    "civilian car": None, "military tank": None, "military truck": None,
}

## 2 · Set up

In [ ]:
!pip install -q ultralytics roboflow

import datetime, json, shutil, sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."

from google.colab import drive
drive.mount("/content/drive")

P = Path(PROJECT)
MODELS = P / "models"
MODELS.mkdir(exist_ok=True)
assert Path(V3_WEIGHTS).exists(), f"Can't find {V3_WEIGHTS}"

# Helper code (frame extraction, labels, dataset building) comes from your GitHub repo.
if not Path("/content/Skynode").exists():
    !git clone -q --depth 1 -b {REPO_BRANCH} {REPO_URL} /content/Skynode
else:
    !git -C /content/Skynode pull -q
sys.path.insert(0, "/content/Skynode/training")
import skynode_data as sd
from ultralytics import YOLO
print("Ready. v3 weights:", V3_WEIGHTS)

## 3 · Part 1: export v3 for the brain
Makes `skynode-v3-<date>.onnx` in **Drive ▸ skynode ▸ models** and prints the lines for `brain/config.toml`.
Class names are stored inside the file, so the dashboard's logger sees `drone` and `aircraft`.

In [ ]:
def export_for_brain(weights, label):
    onnx_path = Path(YOLO(str(weights)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True))
    final = MODELS / f"skynode-{label}-{datetime.date.today().isoformat()}.onnx"
    shutil.copyfile(onnx_path, final)
    print("Saved to Drive:", final)
    print("\nIn brain/config.toml:\n")
    print("[model]")
    print(f'path = "models/{final.name}"')
    print('target_classes = ["drone", "aircraft"]')
    print("\nThen put the file in brain/models/ and run: python -m brain.run --source some_video.mp4")
    return final

v3_onnx = export_for_brain(V3_WEIGHTS, "v3")

## 4 · Part 2: frames from your phone videos, and how v3 does on them *before* any training
v3 predicts on every frame once. Those predictions become (a) the first-guess labels you review, and
(b) a "before" score. In an `aircraft_` clip, a frame where v3 says **drone** is a false alarm; in a clip with a drone, a frame with no drone box is a miss (or the drone was out of view; the sheets will tell you).

In [ ]:
videos = sd.list_videos(PHONE_VIDEOS)
assert videos, f"No videos in {PHONE_VIDEOS}. Upload some (wait for Drive to finish syncing), then run again."

FRAMES = Path("/content/frames")
shutil.rmtree(FRAMES, ignore_errors=True)
frames, hint_of = [], {}
for video in videos:
    got = sd.extract_frames(video, FRAMES, every_s=FRAME_EVERY_S, max_frames=MAX_FRAMES_PER_VIDEO)
    hint = sd.class_hint(video.name, CLASSES)
    for path in got:
        hint_of[path] = hint
    frames += got
    print(f"{video.name}: {len(got)} frames" + (f"  (hint: {hint})" if hint else ""))
print("Total frames:", len(frames))

In [ ]:
def predict_all(model_path, paths, conf=PRELABEL_CONF):
    """-> (boxes, seen): boxes[path] = [(class_name, x1, y1, x2, y2, w, h)], seen[path] = {class names}"""
    model = YOLO(str(model_path))
    boxes, seen = {}, {}
    for start in range(0, len(paths), 16):
        batch = paths[start:start + 16]
        for path, result in zip(batch, model.predict([str(p) for p in batch], imgsz=IMGSZ, conf=conf, verbose=False)):
            height, width = result.orig_shape
            items = [(result.names[int(c)], *xyxy, width, height)
                     for xyxy, c in zip(result.boxes.xyxy.tolist(), result.boxes.cls.tolist())]
            boxes[path], seen[path] = items, {item[0] for item in items}
    return boxes, seen

def rates(seen, paths):
    """How often the model is wrong on clips whose name says what is in them."""
    out = {}
    for hint, wrong in (("aircraft", "drone"), ("drone", "aircraft")):
        subset = [p for p in paths if hint_of[p] == hint]
        if subset:
            out[f"false {wrong} on {hint} clips"] = sum(wrong in seen[p] for p in subset) / len(subset)
            out[f"no {hint} box on {hint} clips"] = sum(hint not in seen[p] for p in subset) / len(subset)
    return out

v3_boxes, v3_seen = predict_all(V3_WEIGHTS, frames)
print("v3 on ALL your phone frames:")
for name, value in rates(v3_seen, frames).items():
    print(f"  {name:32} {value:6.1%}")
print("(Your Roboflow check found 7.5% false-drone on real aircraft. Compare.)")

# First-guess labels: v3's boxes, keeping only the hinted class in clips named after one.
labels = {}
for path in frames:
    lines = []
    for name, x1, y1, x2, y2, w, h in v3_boxes[path]:
        if hint_of[path] in (None, name):
            line = sd.format_label(CLASSES.index(name), x1, y1, x2, y2, w, h)
            if line:
                lines.append(line)
    labels[path] = lines
print("Frames with no box:", sum(1 for v in labels.values() if not v), "of", len(frames))

## 5 · Check the labels
These boxes come from v3, so any mistake v3 makes is baked in unless **you** catch it here. Boxes missing on a real plane or drone do the most harm.
Each picture shows `#number`. In the next cell: **DROP** = discard the frame, **WIPE** = keep the frame but delete its boxes (box on a cloud, or truly empty sky).

In [ ]:
from IPython.display import display
from PIL import Image

PER_SHEET, PAGES_TO_SHOW = 48, 6              # None = show every sheet
pages = (len(frames) + PER_SHEET - 1) // PER_SHEET
shown = pages if PAGES_TO_SHOW is None else min(pages, PAGES_TO_SHOW)
for page in range(shown):
    idx = range(page * PER_SHEET, min(len(frames), (page + 1) * PER_SHEET))
    previews = [sd.draw_preview(frames[i], labels[frames[i]], CLASSES, caption=f"#{i}") for i in idx]
    print(f"Sheet {page + 1} of {pages}")
    display(Image.fromarray(sd.make_contact_sheet(previews, columns=6)[:, :, ::-1]))
if shown < pages:
    print(f"NOT SHOWN: sheets {shown + 1}-{pages} ({len(frames) - shown * PER_SHEET} frames). "
          "Their labels will be used unchecked unless you raise PAGES_TO_SHOW.")

In [ ]:
DROP = []        # frame numbers to throw away, e.g. [3, 17]
WIPE = []        # frame numbers to keep but with no boxes, e.g. [5]

bad = [i for i in DROP + WIPE if not 0 <= i < len(frames)]
assert not bad, f"Not frame numbers: {bad}"
dropped = {frames[i] for i in DROP}
for i in WIPE:
    labels[frames[i]] = []
kept = [p for p in frames if p not in dropped]
print(f"Keeping {len(kept)} frames (dropped {len(dropped)}).")

## 6 · Build the dataset
Whole phone videos are held back for validation (frames from one clip are near-duplicates). Your Roboflow data is added as a sample, with classes folded into drone / aircraft.

In [ ]:
names = sorted(p.name for p in kept)
train_names, val_names = sd.split_frames(names, val_fraction=VAL_FRACTION)
by_name = {p.name: p for p in kept}

DATASET = Path("/content/dataset_v4")
builder = sd.DatasetBuilder(DATASET, CLASSES)
for split, group in (("train", train_names), ("val", val_names)):
    for name in group:
        builder.add(split, by_name[name], labels[by_name[name]])
print(f"Phone frames: {len(train_names)} train, {len(val_names)} validation")

if USE_REPLAY:
    try:
        from google.colab import userdata
        from roboflow import Roboflow
        rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
        rf_dir = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download("yolov8", location="/content/rf_replay")
        import yaml
        old_names = yaml.safe_load(Path(rf_dir.location, "data.yaml").read_text())["names"]
        old_names = [old_names[i] for i in sorted(old_names)] if isinstance(old_names, dict) else list(old_names)
        added, ignored = builder.add_existing(rf_dir.location, old_names, prefix="rf", aliases=ALIASES,
                                              limits=REPLAY_LIMITS, layout="split-first")
        print(f"Replay: added {added} Roboflow images; ignored {ignored} boxes of other classes.")
    except Exception as error:
        print("WARNING: replay skipped (", type(error).__name__, error, ")")
        print("Without it v4 can forget things v3 knew. Fix the key/names above and re-run this cell.")

DATA_YAML = builder.write_yaml()
summary = builder.summary()
for split in ("train", "val"):
    s = summary[split]
    print(f"{split:5} {s['images']:6} images ({s['empty']} empty sky)  boxes: {s['instances']}")
for warning in sd.dataset_warnings(summary):
    print("WARNING:", warning)

## 7 · Train v4 from v3
Checkpoints go to Drive every 5 epochs. Your v3 run folder is not touched.

In [ ]:
last = P / NEW_RUN / "weights" / "last.pt"
if last.exists():
    print("Resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(V3_WEIGHTS).train(
        data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=10, batch=16,   # batch=8 if CUDA runs out of memory
        freeze=FREEZE_LAYERS, lr0=LR0, project=str(P), name=NEW_RUN, exist_ok=True,
        save_period=5, amp=True, workers=2, seed=0,
    )
best_v4 = P / NEW_RUN / "weights" / "best.pt"
print("v4 best weights:", best_v4, "exists:", best_v4.exists())

## 8 · Did it help? (held-out phone videos only)
Numbers on frames v4 trained on would flatter it, so this uses only the validation videos.
Lower is better for the "false" and "no box" rows. With one or two validation clips these numbers are noisy; treat them as a hint.

In [ ]:
val_frames = [by_name[n] for n in val_names]
_, v4_seen = predict_all(best_v4, val_frames)
before, after = rates(v3_seen, val_frames), rates(v4_seen, val_frames)
print(f"{'on held-out phone clips':34}{'v3':>8}{'v4':>8}")
for key in before:
    print(f"{key:34}{before[key]:8.1%}{after.get(key, float('nan')):8.1%}")
if not before:
    print("No clips named drone_* or aircraft_* in the validation set, so there is nothing to score. Name clips after what they show.")
metrics = YOLO(str(best_v4)).val(data=str(DATA_YAML), imgsz=IMGSZ, split="val", verbose=False)
print(f"\nv4 validation (phone + Roboflow sample)  mAP50 = {metrics.box.map50:.3f}   mAP50-95 = {metrics.box.map:.3f}")

## 9 · Export v4 for the brain

In [ ]:
v4_onnx = export_for_brain(best_v4, "v4")
from google.colab import files
files.download(str(v4_onnx))

### After this
- Test whichever model scores better on a video neither saw: `python -m brain.run --source new_clip.mp4`.
- More phone clips in different light and weather beat more epochs. Add them to `phone_videos` and run again (new run name in Settings, e.g. `drone_v5`, starting from your best weights).
- Never paste your Roboflow key into a cell. Keep it in the Colab Secrets panel.